Just a dev notebook to develop the algorithm. Will export the finals into a py file for execution. 

In [22]:
# Automatically reload imported modules when their source files change.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
# Import the data
import pandas as pd
import numpy as np 
from utils.data_loading import load_participant_details

participant_details = load_participant_details('./raw_data/participant_details.xlsx')

participant = participant_details[0]
path = participant['filename']
data = np.loadtxt(f'./raw_data/{path}', delimiter='\t', skiprows=1, usecols=range(1, 19))

In [24]:
# Sum Humerothoracic rotation
from utils.kinematics.cumulative import calculate_arm_rotations
left_rotation, right_rotation = calculate_arm_rotations(data)
participant['left']['humerothoracic_rotation'] = left_rotation
participant['right']['humerothoracic_rotation'] = right_rotation

print(f"Left arm total rotation: {left_rotation:.2f} radians")
print(f"Right arm total rotation: {right_rotation:.2f} radians")

Left arm total rotation: 354272.47 radians
Right arm total rotation: 359999.97 radians


In [25]:
# Sum motion about each axis
from utils.kinematics.individual_axes import calculate_cumulative_axis_motion
left_axes = calculate_cumulative_axis_motion(data, 'L')
right_axes = calculate_cumulative_axis_motion(data, 'R')
participant['left']['cumulative_axes'] = left_axes
participant['right']['cumulative_axes'] = right_axes
print(f"Left arm cumulative axis motion (radians): {left_axes}")
print(f"Right arm cumulative axis motion (radians): {right_axes}")

ValueError: relative_rotations must be orthonormal rotation matrices

In [ ]:
"""
Getting errors about orthonormality. Tells me we might have data drift (or maybe just a bug).

Go through and see how much error is present at each timestamp. Log the values over time. 

```
I = np.eye(3)

for i, R in enumerate(matrices):
    ortho_error = np.linalg.norm(R.T @ R - I)
    det_error = np.abs(np.linalg.det(R) - 1)

    if ortho_error > 1e-6 or det_error > 1e-6:
        print(i, ortho_error, det_error)
```

| Error        | Meaning                      |
| ------------ | ---------------------------- |
| ~1e-12       | fine (floating point noise)  |
| 1e-6 to 1e-3 | mild drift / sensor noise    |
| >1e-3        | serious corruption           |
| >> 1         | not a rotation matrix at all |


Make decisions from there.
"""

In [ ]:
# extract the filename, arm, and handedness
# for each arm
    # sum humerothoracic rotation
    # sum glenoohumeral rotation (later)
    # bin %time spent in each region of elevation and PoE